# Backtest, model comparison, and the Pareto frontier

This notebook covers the second half: how the ten models differ, the
walk-forward protocol, the full comparison, the tuned-vs-default check, the
cross-zone robustness run, and the per-regime Pareto frontier that closes the
story.


## The four families

The roster spans four families. The classical models (SARIMA, ETS) and the
regularized linear model (LEAR) fit the price series alone or with a small
exogenous set. The gradient-boosted trees (LightGBM, XGBoost, CatBoost) train
three quantile regressors on the full feature matrix. The deep models
(N-BEATS, TFT) are neural; N-BEATS is price-only and TFT consumes the full
feature set. The foundation models (Chronos-2, TimesFM 2.5) are pretrained and
zero-shot, Chronos-2 with covariates and TimesFM price-only.


Each model is routed to the feature set it can consume. Price-only
models (SARIMA, ETS, LEAR, N-BEATS, TimesFM) see group 1. Full-features models
(LightGBM, XGBoost, CatBoost, TFT, Chronos-2) see groups 1 and 3 through 7,
with group 2 excluded by the as-of rule.


## The backtest protocol

The comparison uses an expanding walk-forward window with yearly refits and a
7-day purge between training and the forecast day. The hourly regime runs
2023-01-01 to 2025-09-30 (three folds); the 15-minute regime runs from the
2025-10-01 market switch onward (one fold). The primary metric is mean CRPS,
with pinball loss and MAE reported alongside and a seasonal-naive skill score.


In [ ]:
from forecast_pipeline.snapshot import SNAPSHOT_DIR, headline_ranking, load_results

tables = load_results(SNAPSHOT_DIR)
results = tables["results"]
ranking = headline_ranking(results)
print("headline ranking (mean CRPS, lower is better):")
print(ranking.to_string(index=False))


## The headline comparison

The gradient-boosted trees dominate the accuracy-per-compute frontier.
Chronos-2 edges the trees on mean CRPS by about 0.5% but costs roughly 1500
times the compute. The deep and classical models are dominated on both axes.


In [ ]:
pareto = tables["pareto_table"]
print("pareto table (frontier flag computed within each regime):")
cols = ["model", "family", "regime", "n_folds", "CRPS", "total_compute_s", "pareto_optimal"]
print(pareto[cols].sort_values(["regime", "CRPS"]).to_string(index=False) if not pareto.empty else "(empty)")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not pareto.empty:
    for regime, grp in pareto.groupby("regime"):
        front = grp[grp["pareto_optimal"]]
        dominated = grp[~grp["pareto_optimal"]]
        plt.scatter(np.log1p(dominated["total_compute_s"]), dominated["CRPS"],
                    label="dominated", alpha=0.6)
        plt.scatter(np.log1p(front["total_compute_s"]), front["CRPS"],
                    label="frontier", s=90)
        plt.xlabel("log1p(total compute, s)")
        plt.ylabel("mean CRPS")
        plt.title(f"Pareto frontier, {regime} regime")
        plt.legend()
        plt.show()


The frontier flag is computed within each regime (ticket 20), so a
model that is cheap and accurate in one regime does not dominate a model in
the other. Earlier exports pooled the two regimes and mis-flagged the
frontier; the corrected table is the one shown here.


## Tuned versus default

A secondary comparison tunes the best model per family and reports the delta
against the pinned default. The default table stays the headline. SARIMA is
tuned by an AIC/BIC order grid, CatBoost and N-BEATS by bounded Optuna, and
Chronos-2 by a context-length sweep (its weights are pretrained and are never
tuned).


In [ ]:
tuned = tables["tuned_table"]
print("tuned-vs-default (delta = tuned - default; negative is better):")
print(tuned.to_string(index=False) if not tuned.empty else "(empty)")


## Cross-zone robustness

The headline runs on SE3. A reduced run takes the frontier (the three trees)
plus Chronos-2 across SE1 through SE4 at default configs, hourly regime only,
to check whether the frontier generalises and whether Chronos-2's accuracy
edge survives outside SE3.


In [ ]:
cross = tables["cross_zone_summary"]
print("per-zone ranking of the four frontier models:")
print(cross.to_string(index=False) if not cross.empty else "(empty)")


## The final answer

The per-regime Pareto frontier is the closing result: the gradient-boosted
trees sit on the accuracy-per-compute frontier in both regimes, Chronos-2
holds the accuracy edge at a large compute premium, and the deep and classical
models are dominated. The snapshot directory holds every table and matrix
needed to reproduce these figures offline, with no API key.
